# Notebook 1: Monte Carlo Portfolio Risk Simulation
**Project:** Monte Carlo Risk & Derivatives Pricing
**Author:** Niraj Neupane | github.com/nirajneupane17

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy import stats
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(13,6),'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})
returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
weights = np.array([0.25, 0.25, 0.20, 0.15, 0.15])
port = returns @ weights
print(f'Loaded {len(port):,} observations')


Loaded 2,609 observations


## 1. GBM Portfolio Simulation

In [2]:
from mc_simulation import simulate_gbm_portfolio, mc_var_es
paths = simulate_gbm_portfolio(port, n_sims=10000, horizon=252)
print(f'Simulated paths shape: {paths.shape}')
print(f'Mean final value : ${paths[:,-1].mean():.2f}')
print(f'5th pct final    : ${np.percentile(paths[:,-1],5):.2f}')
print(f'95th pct final   : ${np.percentile(paths[:,-1],95):.2f}')

Simulated paths shape: (10000, 252)
Mean final value : $99.98
5th pct final    : $98.76
95th pct final   : $101.19


In [3]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
ax1=axes[0]
for i in range(200):
    ax1.plot(paths[i], linewidth=0.3, alpha=0.3, color='#185FA5')
ax1.plot(paths[:200].mean(axis=0), color='#E24B4A', linewidth=2, label='Mean path')
ax1.plot(np.percentile(paths[:200],5,axis=0), color='#F57C00', linewidth=1.5, linestyle='--', label='5th percentile')
ax1.plot(np.percentile(paths[:200],95,axis=0), color='#1D9E75', linewidth=1.5, linestyle='--', label='95th percentile')
ax1.set_title('Monte Carlo Portfolio Simulation (200 of 10,000 paths)')
ax1.set_xlabel('Trading Days'); ax1.set_ylabel('Portfolio Value ($)'); ax1.legend(fontsize=9)
final=paths[:,-1]
ax2=axes[1]
ax2.hist(final, bins=80, color='#185FA5', alpha=0.7, edgecolor='white', linewidth=0.3, density=True)
ax2.axvline(np.percentile(final,5), color='#E24B4A', linewidth=2, linestyle='--', label=f'5th pct=${np.percentile(final,5):.1f}')
ax2.axvline(np.percentile(final,50), color='#1D9E75', linewidth=2, linestyle='--', label=f'Median=${np.percentile(final,50):.1f}')
ax2.set_title('Distribution of Portfolio Values at 1-Year Horizon')
ax2.set_xlabel('Portfolio Value ($)'); ax2.legend(fontsize=9)
plt.tight_layout(); plt.savefig('../results/01_mc_portfolio_paths.png',dpi=150,bbox_inches='tight')
plt.show()

## 2. MC VaR and Expected Shortfall

In [4]:
mc_res = mc_var_es(returns, weights, n_sims=10000)
print('Monte Carlo VaR and Expected Shortfall:')
print(mc_res.round(4))
mc_res.to_csv('../results/mc_var_es_results.csv')

Monte Carlo VaR and Expected Shortfall:
            MC_VaR   MC_ES  ES_VaR_ratio
confidence                              
90%         0.0097  0.0132         1.368
95%         0.0124  0.0155         1.250
99%         0.0174  0.0201         1.155
99%         0.0192  0.0219         1.138


In [5]:
sim_port = np.random.multivariate_normal(returns.mean().values, returns.cov().values, 10000) @ weights
fig,axes=plt.subplots(1,2,figsize=(14,5))
ax1=axes[0]
ax1.hist(sim_port,bins=100,color='#185FA5',alpha=0.7,edgecolor='white',linewidth=0.2,density=True)
for cl,c in zip([0.95,0.99,0.995],["#F57C00","#E24B4A","#7B1FA2"]):
    v=abs(np.percentile(sim_port,(1-cl)*100))
    ax1.axvline(-v,color=c,linewidth=1.8,linestyle='--',label=f'{int(cl*100)}% VaR={v:.4f}')
ax1.set_title('Monte Carlo Return Distribution (10,000 sims)'); ax1.legend(fontsize=8)
ax2=axes[1]
sizes=[100,500,1000,2000,5000,10000]
conv=[abs(np.percentile(sim_port[:n],1)) for n in sizes]
ax2.plot(sizes,conv,'o-',color='#E24B4A',linewidth=2,markersize=8)
ax2.axhline(conv[-1],color='gray',linewidth=1,linestyle='--',alpha=0.6)
ax2.set_title('99% VaR Convergence'); ax2.set_xscale('log')
plt.tight_layout(); plt.savefig('../results/02_mc_var_distribution.png',dpi=150,bbox_inches='tight')
plt.show()